# Проект. Модель для показа рекламных предложений

В этом проекте вам необходимо реализовать мониторинг модели с помощью Prometheus и Grafana. 

Этапы, которые нужно пройти:
- Создание модели.
- Скраппинг метрик модели с помощью Prometheus.
- Подключение Prometheus к Grafana.
- Отрисовка дашборда.

С нуля писать взаимодействие между всеми этими компонентами довольно сложно. Будет очень здорово, если у вас получится отобразить хотя бы score модели на дашборде Grafana, который будет передаваться через Prometheus. 

Если же вы хотите более красочный дашборд, то можно пойти немного другим путём и использовать уже заранее созданные взаимодействия между этими компонентами в Kubernetes. Подробная инструкция

Критерии проверки

| Критерий | Баллы |
| -------------|------|
| Создана модель | 2 |
| Метрики модели передаются в Prometheus | 2 |
| На дашборде отображены метрики модели | 4 |


 Модель настроена так, что её можно дообучать 

In [ ]:
# импорт библиотек

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from catboost import CatBoostRegressor, Pool
import joblib
import json
from datetime import datetime


In [2]:
# Загрузка данных
df = pd.read_csv('winequality-red.csv', sep=';')
print(f"Данные загружены: {df.shape[0]} строк")

X = df.drop('quality', axis=1)
y = df['quality']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


Данные загружены: 1599 строк


In [ ]:
# МОДЕЛЬ 
model_path = 'wine_quality_model.cbm'
metrics_path = 'model_metrics.json'

# создаем модель
model = CatBoostRegressor(iterations=800, learning_rate=0.08, depth=6, 
                              loss_function='RMSE', random_seed=42, verbose=100)

# Обучение
train_pool = Pool(X_train, y_train)
test_pool = Pool(X_test, y_test)

model.fit(train_pool, eval_set=test_pool, early_stopping_rounds=50)

# Метрики
preds = model.predict(X_test)
metrics = {
    "mae": float(mean_absolute_error(y_test, preds)),
    "rmse": float(mean_squared_error(y_test, preds)),
    "r2": float(r2_score(y_test, preds)),
    "last_trained": datetime.now().isoformat(),
    "n_samples": len(X_train)
}

print("\n📊 Результаты модели:")
print(f"MAE:  {metrics['mae']:.3f}")
print(f"RMSE: {metrics['rmse']:.3f}")
print(f"R²:   {metrics['r2']:.3f}")

# Сохраняем
model.save_model(model_path)
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"✅ Модель сохранена в {model_path}")


🆕 Создаём новую модель...
0:	learn: 0.7857693	test: 0.7930496	best: 0.7930496 (0)	total: 3.04ms	remaining: 2.43s
100:	learn: 0.5171247	test: 0.5848967	best: 0.5848967 (100)	total: 138ms	remaining: 957ms
200:	learn: 0.4277954	test: 0.5716395	best: 0.5716395 (200)	total: 275ms	remaining: 820ms
300:	learn: 0.3494826	test: 0.5616423	best: 0.5616423 (300)	total: 425ms	remaining: 705ms
400:	learn: 0.2882119	test: 0.5519842	best: 0.5519842 (400)	total: 560ms	remaining: 557ms
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5505958706
bestIteration = 414

Shrink model to first 415 iterations.

📊 Результаты модели:
MAE:  0.427
RMSE: 0.303
R²:   0.536
✅ Модель сохранена в wine_quality_model.cbm


In [5]:
#  ФУНКЦИИ 
def predict_quality(features_dict):
    """Предсказание по словарю"""
    df_pred = pd.DataFrame([features_dict])
    pred = model.predict(df_pred)[0]
    return round(float(pred), 2)

def retrain_model(new_df):
    """Дообучение на новых данных"""
    global model
    X_new = new_df.drop('quality', axis=1)
    y_new = new_df['quality']
    
    print(f"🔄 Дообучаем на {len(new_df)} новых примерах...")
    model.fit(Pool(X_new, y_new), init_model=model)
    model.save_model(model_path)
    print("✅ Дообучение завершено!")
    return predict_quality(X_new.iloc[0].to_dict())

# Пример использования
example = {
    "fixed acidity": 7.4, "volatile acidity": 0.7, "citric acid": 0.0,
    "residual sugar": 1.9, "chlorides": 0.076, "free sulfur dioxide": 11.0,
    "total sulfur dioxide": 34.0, "density": 0.9978, "pH": 3.51,
    "sulphates": 0.56, "alcohol": 9.4
}

print("\n🎯 Пример предсказания:", predict_quality(example))


🎯 Пример предсказания: 5.03
